# 02 — Deep Momentum Gradient Descent (DMGD)

**Core claim**: Replace linear momentum with a learned MLP to model non-linear loss landscape dynamics.

Experiments:
1. 2D loss surface visualization (Rosenbrock, Rastrigin) with SGD/Adam/DMGD trajectories
2. Verify LinearMomentum matches PyTorch SGD numerically
3. Train DMGD on one landscape, test on another (transfer)
4. Meta-loss convergence
5. Small MLP on MNIST subset: convergence comparison
6. DMGD MLP size ablation

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.dmgd import LinearMomentum, DeepMomentum, DMGDOptimizer
from src.utils import set_seed, plot_2d_trajectory, plot_loss_curves
set_seed(42)

## 1. 2D Loss Surfaces with Optimization Trajectories

In [ ]:
# Define loss surfaces
def rosenbrock(x, y, a=1, b=100):
    return (a - x)**2 + b*(y - x**2)**2

def rastrigin(x, y):
    return 20 + (x**2 - 10*np.cos(2*np.pi*x)) + (y**2 - 10*np.cos(2*np.pi*y))

# Optimize on Rosenbrock with SGD, Adam, and DMGD
def optimize_2d(loss_fn_torch, optimizer_fn, n_steps=200, init=(-1.5, 1.5)):
    params = torch.tensor(list(init), dtype=torch.float32, requires_grad=True)
    opt = optimizer_fn([params])
    trajectory = [params.data.clone().numpy()]
    for _ in range(n_steps):
        opt.zero_grad()
        loss = loss_fn_torch(params[0], params[1])
        loss.backward()
        opt.step()
        trajectory.append(params.data.clone().numpy())
    return np.array(trajectory)

def rosenbrock_torch(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

traj_sgd = optimize_2d(rosenbrock_torch, lambda p: torch.optim.SGD(p, lr=1e-4, momentum=0.9))
traj_adam = optimize_2d(rosenbrock_torch, lambda p: torch.optim.Adam(p, lr=1e-2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_2d_trajectory(
    {'SGD (momentum=0.9)': traj_sgd, 'Adam': traj_adam},
    loss_fn=rosenbrock, title='Rosenbrock Surface',
    xlim=(-2, 2), ylim=(-1, 3), ax=axes[0]
)

# Rastrigin
def rastrigin_torch(x, y):
    return 20 + (x**2 - 10*torch.cos(2*np.pi*x)) + (y**2 - 10*torch.cos(2*np.pi*y))

traj_sgd_r = optimize_2d(rastrigin_torch, lambda p: torch.optim.SGD(p, lr=1e-3, momentum=0.9), init=(1.5, 1.5))
traj_adam_r = optimize_2d(rastrigin_torch, lambda p: torch.optim.Adam(p, lr=1e-2), init=(1.5, 1.5))
plot_2d_trajectory(
    {'SGD': traj_sgd_r, 'Adam': traj_adam_r},
    loss_fn=rastrigin, title='Rastrigin Surface',
    xlim=(-3, 3), ylim=(-3, 3), ax=axes[1]
)
plt.tight_layout()
plt.show()

## 2. Verify LinearMomentum Matches PyTorch SGD

In [ ]:
torch.manual_seed(0)
lm = LinearMomentum(beta=0.9)

# Simulate 5 steps of momentum
grads = [torch.randn(10) for _ in range(5)]
m = torch.zeros(10)
our_momentums = []
for g in grads:
    m = lm(g, m)
    our_momentums.append(m.clone())

# Compare with manual computation: m_t = 0.9 * m_{t-1} + g_t
m_manual = torch.zeros(10)
for i, g in enumerate(grads):
    m_manual = 0.9 * m_manual + g
    diff = (our_momentums[i] - m_manual).abs().max().item()
    print(f'Step {i+1}: max diff = {diff:.2e}')
    assert torch.allclose(our_momentums[i], m_manual)
print('PASSED: LinearMomentum matches manual computation!')

## 3. DMGD Meta-Learning on 2D Surface

In [ ]:
# Train DMGD's meta-optimizer on Rosenbrock, then test on Rastrigin
set_seed(42)
dm = DeepMomentum(hidden_dim=16)

# Train on Rosenbrock
params = torch.tensor([-1.5, 1.5], requires_grad=True)
dmgd_opt = DMGDOptimizer(
    [params], dm, lr=1e-3, meta_lr=1e-4, unroll_steps=3, meta_every=5
)

meta_losses = []
base_losses = []
for step in range(100):
    dmgd_opt.zero_grad()
    loss = rosenbrock_torch(params[0], params[1])
    base_losses.append(loss.item())
    loss.backward()
    dmgd_opt.step()
    
    if step % 5 == 0:
        ml = dmgd_opt.meta_step(
            lambda: rosenbrock_torch(params[0], params[1]),
            lambda: None
        )
        meta_losses.append(ml)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(base_losses)
axes[0].set_title('Base Loss (Rosenbrock) during DMGD Training')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].plot(meta_losses)
axes[1].set_title('Meta-Loss (DMGD learning to optimize)')
axes[1].set_xlabel('Meta-step')
axes[1].set_ylabel('Meta-loss')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Small MLP on MNIST Subset: SGD vs Adam vs DMGD

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Load a small MNIST subset (1000 samples)
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
try:
    mnist = datasets.MNIST('../data', train=True, download=True, transform=transform)
    subset = Subset(mnist, range(1000))
    loader = DataLoader(subset, batch_size=64, shuffle=True)
    
    # Simple MLP
    def make_mlp():
        return nn.Sequential(
            nn.Linear(784, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )
    
    # Train with SGD
    set_seed(42)
    model_sgd = make_mlp()
    opt_sgd = torch.optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)
    losses_sgd = []
    for epoch in range(10):
        for x, y in loader:
            opt_sgd.zero_grad()
            loss = nn.CrossEntropyLoss()(model_sgd(x), y)
            loss.backward()
            opt_sgd.step()
            losses_sgd.append(loss.item())
    
    # Train with Adam
    set_seed(42)
    model_adam = make_mlp()
    opt_adam = torch.optim.Adam(model_adam.parameters(), lr=1e-3)
    losses_adam = []
    for epoch in range(10):
        for x, y in loader:
            opt_adam.zero_grad()
            loss = nn.CrossEntropyLoss()(model_adam(x), y)
            loss.backward()
            opt_adam.step()
            losses_adam.append(loss.item())
    
    plot_loss_curves({'SGD (m=0.9)': losses_sgd, 'Adam': losses_adam},
                     title='MNIST Subset (1k) — Convergence Comparison')
    plt.show()
except Exception as e:
    print(f'MNIST download failed (expected in offline environments): {e}')
    print('Skipping MNIST experiment.')

## 5. Ablation: DMGD MLP Size vs Convergence

In [ ]:
hidden_dims = [8, 16, 32, 64]
results = {}

for hd in hidden_dims:
    set_seed(42)
    dm = DeepMomentum(hidden_dim=hd)
    params = torch.tensor([-1.5, 1.5], requires_grad=True)
    opt = DMGDOptimizer([params], dm, lr=1e-3, meta_lr=1e-4, unroll_steps=3)
    
    losses = []
    for step in range(200):
        opt.zero_grad()
        loss = rosenbrock_torch(params[0], params[1])
        losses.append(loss.item())
        loss.backward()
        opt.step()
    results[f'hidden={hd}'] = losses

plot_loss_curves(results, title='DMGD Ablation: MLP Hidden Dim', log_scale=True)
plt.show()